# Statcast-Era Team Forecast -- Kaggle GPU Run

**Track**: Statcast-era (2015+, deeper feature depth). All 30 teams,
pooled into one shared `TeamPanelNet`. See
`../../statcast_era_smoke.ipynb` for the CPU-scale pipeline-correctness
check this scales up from, and the plan document for full rationale
(team-level `hfTeam` filter confirmed live, regime-indicator features for
2020/2021/2023/2025 confounds, leave-team-out internal validation given
how few years exist per team).

**Kaggle settings required**: GPU (T4x2) and internet both enabled.


In [ ]:

import json
import os
import time
import numpy as np
import pandas as pd
import requests
import torch
from sklearn.metrics import mean_squared_error, r2_score

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

SAVANT_URL = "https://baseballsavant.mlb.com/statcast_search/csv"
BASE_URL = "https://statsapi.mlb.com/api/v1"
HEADERS = {"User-Agent": "Mozilla/5.0 (MLB-Analytics-Dashboard-Telemetry/1.0; AustinKuo)"}


def safe_float(val, default=float("nan")):
    try:
        s = str(val).strip()
        return default if s in ("-.--", "---", "", "INF", "inf") else float(s)
    except (ValueError, TypeError):
        return default


def safe_int(val, default=0):
    try:
        return int(float(val))
    except (ValueError, TypeError):
        return default


def regime_flags(year):
    """Binary indicators for known structural breaks in the Statcast era,
    so the model sees these as explicit signal rather than having to infer
    them from raw metric trends. See the plan's rationale for each:
    2020 (COVID 60-game season), 2021 (sticky-substance crackdown, mid-season
    but treated as a full-season flag at this granularity), 2023 (pitch
    clock/bigger bases/shift limits), 2025 (further shift-rule tightening).
    """
    return {
        "is_covid_2020": 1.0 if year == 2020 else 0.0,
        "post_sticky_crackdown": 1.0 if year >= 2021 else 0.0,
        "pitch_clock_era": 1.0 if year >= 2023 else 0.0,
        "shift_limit_tightened_2025": 1.0 if year >= 2025 else 0.0,
    }

print("Environment ready.")


In [ ]:

TRACK = "statcast_era"
ENVIRONMENT = "kaggle_gpu"

ALL_TEAMS = [{'team_id': 108, 'abbreviation': 'LAA'}, {'team_id': 109, 'abbreviation': 'ARI'}, {'team_id': 110, 'abbreviation': 'BAL'}, {'team_id': 111, 'abbreviation': 'BOS'}, {'team_id': 112, 'abbreviation': 'CHC'}, {'team_id': 113, 'abbreviation': 'CIN'}, {'team_id': 114, 'abbreviation': 'CLE'}, {'team_id': 115, 'abbreviation': 'COL'}, {'team_id': 116, 'abbreviation': 'DET'}, {'team_id': 117, 'abbreviation': 'HOU'}, {'team_id': 118, 'abbreviation': 'KC'}, {'team_id': 119, 'abbreviation': 'LAD'}, {'team_id': 120, 'abbreviation': 'WSH'}, {'team_id': 121, 'abbreviation': 'NYM'}, {'team_id': 133, 'abbreviation': 'ATH'}, {'team_id': 134, 'abbreviation': 'PIT'}, {'team_id': 135, 'abbreviation': 'SD'}, {'team_id': 136, 'abbreviation': 'SEA'}, {'team_id': 137, 'abbreviation': 'SF'}, {'team_id': 138, 'abbreviation': 'STL'}, {'team_id': 139, 'abbreviation': 'TB'}, {'team_id': 140, 'abbreviation': 'TEX'}, {'team_id': 141, 'abbreviation': 'TOR'}, {'team_id': 142, 'abbreviation': 'MIN'}, {'team_id': 143, 'abbreviation': 'PHI'}, {'team_id': 144, 'abbreviation': 'ATL'}, {'team_id': 145, 'abbreviation': 'CWS'}, {'team_id': 146, 'abbreviation': 'MIA'}, {'team_id': 147, 'abbreviation': 'NYY'}, {'team_id': 158, 'abbreviation': 'MIL'}]

STATCAST_ERA_START_YEAR = 2015
LATEST_COMPLETE_SEASON = 2024   # update as seasons close
HOLDOUT_YEARS = 1
TRAIN_START_YEAR = STATCAST_ERA_START_YEAR
TRAIN_END_YEAR = LATEST_COMPLETE_SEASON - HOLDOUT_YEARS
HOLDOUT_START_YEAR = TRAIN_END_YEAR + 1
HOLDOUT_END_YEAR = LATEST_COMPLETE_SEASON
FORECAST_END_YEAR = LATEST_COMPLETE_SEASON + 2

TARGETS = ["win_pct", "runs_scored_per_game", "runs_allowed_per_game", "team_ops", "team_era"]
STATCAST_FEATURES = ["team_avg_exit_velocity", "team_hard_hit_pct", "team_barrel_pct", "team_xba",
                      "team_csw_pct", "team_whiff_pct", "team_chase_pct", "team_avg_velocity"]
REGIME_FLAG_NAMES = ["is_covid_2020", "post_sticky_crackdown", "pitch_clock_era", "shift_limit_tightened_2025"]

HIDDEN = (16, 8)
EMBED_DIM = 4
DROPOUT = 0.35
EPOCHS = 300
EARLY_STOPPING_PATIENCE = 20
LR = 1e-3
WEIGHT_DECAY = 1e-3

print(f"train {TRAIN_START_YEAR}-{TRAIN_END_YEAR}, holdout {HOLDOUT_START_YEAR}-{HOLDOUT_END_YEAR}")


## Data fetch

`hfTeam=<ABBR>|` combined with `player_type=batter`/`pitcher` scopes
results to that team's own hitters/pitchers only -- confirmed live this
session, no extra home/away filtering needed. One CSV pull per
team-season per side; each can run tens of MB, so this cell is slow.


In [ ]:

CSW_DESCRIPTIONS = {"swinging_strike", "swinging_strike_blocked", "missed_bunt", "called_strike"}
SWING_DESCRIPTIONS = {"hit_into_play", "foul", "swinging_strike_blocked", "foul_bunt", "foul_tip",
                       "swinging_strike", "missed_bunt"}
WHIFF_DESCRIPTIONS = {"swinging_strike", "swinging_strike_blocked", "missed_bunt"}
OUT_OF_ZONE_CODES = {"11", "12", "13", "14"}
HARD_HIT_LAUNCH_SPEED_MPH = 95.0
BARREL_LAUNCH_SPEED_ANGLE_CODE = "6"


def fetch_savant_team_season(abbreviation, year, player_type):
    import csv, io
    params = {"all": "true", "hfGT": "R|", "type": "details", "hfSea": f"{year}|",
              "player_type": player_type, "game_date_gt": f"{year}-01-01",
              "game_date_lt": f"{year}-12-31", "hfTeam": f"{abbreviation}|"}
    try:
        r = requests.get(SAVANT_URL, params=params, timeout=180)
        r.raise_for_status()
        text = r.content.decode("utf-8-sig")
        return list(csv.DictReader(io.StringIO(text))) if text.strip() else []
    except Exception:
        return []


def team_hitter_statcast(rows):
    batted = [r for r in rows if r.get("type") == "X"]
    if not batted:
        return {}
    exit_velos = [v for v in (safe_float(r.get("launch_speed"), None) for r in batted) if v is not None]
    xbas = [v for v in (safe_float(r.get("estimated_ba_using_speedangle"), None) for r in batted) if v is not None]
    barrels = sum(1 for r in batted if r.get("launch_speed_angle") == BARREL_LAUNCH_SPEED_ANGLE_CODE)
    return {
        "team_xba": (sum(xbas) / len(xbas)) if xbas else None,
        "team_avg_exit_velocity": (sum(exit_velos) / len(exit_velos)) if exit_velos else None,
        "team_hard_hit_pct": (sum(1 for v in exit_velos if v >= HARD_HIT_LAUNCH_SPEED_MPH) / len(exit_velos)) if exit_velos else None,
        "team_barrel_pct": barrels / len(batted),
    }


def team_pitcher_statcast(rows):
    if not rows:
        return {}
    total = len(rows)
    csw = sum(1 for r in rows if r.get("description") in CSW_DESCRIPTIONS)
    swings = sum(1 for r in rows if r.get("description") in SWING_DESCRIPTIONS)
    whiffs = sum(1 for r in rows if r.get("description") in WHIFF_DESCRIPTIONS)
    out_of_zone = [r for r in rows if r.get("zone") in OUT_OF_ZONE_CODES]
    oz_swings = sum(1 for r in out_of_zone if r.get("description") in SWING_DESCRIPTIONS)
    velocities = [v for v in (safe_float(r.get("release_speed"), None) for r in rows) if v is not None]
    return {
        "team_csw_pct": csw / total,
        "team_whiff_pct": (whiffs / swings) if swings else None,
        "team_chase_pct": (oz_swings / len(out_of_zone)) if out_of_zone else None,
        "team_avg_velocity": (sum(velocities) / len(velocities)) if velocities else None,
    }


def fetch_statcast_era_frame(teams, start_year, end_year):
    rows = []
    for team in teams:
        for year in range(start_year, end_year + 1):
            hitter_rows = fetch_savant_team_season(team["abbreviation"], year, "batter")
            pitcher_rows = fetch_savant_team_season(team["abbreviation"], year, "pitcher")
            stats = {**team_hitter_statcast(hitter_rows), **team_pitcher_statcast(pitcher_rows)}
            if stats and all(v is not None for v in stats.values()):
                stats.update(team_id=team["team_id"], year=year)
                rows.append(stats)
    return pd.DataFrame(rows)


t0 = time.time()
statcast_raw = fetch_statcast_era_frame(ALL_TEAMS, TRAIN_START_YEAR, HOLDOUT_END_YEAR)
print(f"Fetched {len(statcast_raw)} team-year Statcast rows in {time.time()-t0:.0f}s.")


In [ ]:

def fetch_team_season_stat_row(team_id, year):
    try:
        hit = requests.get(f"{BASE_URL}/teams/{team_id}/stats",
                            params={"stats": "season", "group": "hitting", "season": year},
                            headers=HEADERS, timeout=15).json()
        pit = requests.get(f"{BASE_URL}/teams/{team_id}/stats",
                            params={"stats": "season", "group": "pitching", "season": year},
                            headers=HEADERS, timeout=15).json()
    except Exception:
        return None
    hit_splits = hit.get("stats", [{}])[0].get("splits", [])
    pit_splits = pit.get("stats", [{}])[0].get("splits", [])
    if not hit_splits or not pit_splits:
        return None
    h, p = hit_splits[0]["stat"], pit_splits[0]["stat"]
    games = safe_int(p.get("gamesPlayed"))
    if games == 0:
        return None
    return {"games": games, "wins": safe_int(p.get("wins")), "runs_scored": safe_int(h.get("runs")),
            "runs_allowed": safe_int(p.get("runs")), "team_ops": safe_float(h.get("ops")),
            "team_era": safe_float(p.get("era"))}


target_rows = []
for team in ALL_TEAMS:
    for year in range(TRAIN_START_YEAR, HOLDOUT_END_YEAR + 1):
        row = fetch_team_season_stat_row(team["team_id"], year)
        if row is None:
            continue
        row.update(team_id=team["team_id"], year=year)
        target_rows.append(row)
targets_df = pd.DataFrame(target_rows)
targets_df["win_pct"] = targets_df["wins"] / targets_df["games"]
targets_df["runs_scored_per_game"] = targets_df["runs_scored"] / targets_df["games"]
targets_df["runs_allowed_per_game"] = targets_df["runs_allowed"] / targets_df["games"]

raw = targets_df.merge(statcast_raw, on=["team_id", "year"], how="inner")
print(f"{len(raw)} merged team-year rows.")


In [ ]:

def add_lag_rolling_statcast_features(df):
    df = df.sort_values(["team_id", "year"]).copy()
    for feat in STATCAST_FEATURES:
        grp = df.groupby("team_id")[feat]
        df[f"lag_1_{feat}"] = grp.shift(1)
    return df


team_ids_sorted = sorted(t["team_id"] for t in ALL_TEAMS)
TEAM_EMBED_INDEX = {tid: i for i, tid in enumerate(team_ids_sorted)}

featured = add_lag_rolling_statcast_features(raw)
featured["team_embedding_index"] = featured["team_id"].map(TEAM_EMBED_INDEX)
featured["year_norm"] = (featured["year"] - STATCAST_ERA_START_YEAR) / 15.0
regime_cols = pd.DataFrame([regime_flags(y) for y in featured["year"]], index=featured.index)
featured = pd.concat([featured, regime_cols], axis=1)

FEATURE_COLS = ["year_norm"] + STATCAST_FEATURES + [f"lag_1_{f}" for f in STATCAST_FEATURES] + REGIME_FLAG_NAMES

excluded_rows = featured[featured["is_covid_2020"] == 1.0][["team_id", "year"]].to_dict("records")
featured = featured.dropna(subset=FEATURE_COLS).reset_index(drop=True)
training_pool = featured[featured["is_covid_2020"] == 0.0].reset_index(drop=True)
print(f"{len(featured)} feature-complete rows ({len(training_pool)} trainable after excluding 2020).")


## Split, model, training

Leave-team-out internal validation: 6 of 30 teams (20%) held out from
`fit_df`, all years -- see the plan's rationale for why this differs from
what the real holdout measures.


In [ ]:

import torch
import torch.nn as nn


class TeamPanelNet(nn.Module):
    """Feedforward net over engineered lag/rolling features, with team
    identity as a learned embedding and one shared trunk predicting all
    targets at once (a regularizer in itself at this row count, since the
    targets are correlated -- wins/runs-scored/runs-allowed/OPS/ERA all
    move together).
    """

    def __init__(self, n_teams, n_numeric_features, n_targets, embed_dim=8, hidden=(64, 32), dropout=0.2):
        super().__init__()
        self.team_embed = nn.Embedding(n_teams, embed_dim)
        layers, in_dim = [], embed_dim + n_numeric_features
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        self.trunk = nn.Sequential(*layers)
        self.head = nn.Linear(in_dim, n_targets)

    def forward(self, team_idx, x_numeric):
        z = torch.cat([self.team_embed(team_idx), x_numeric], dim=-1)
        return self.head(self.trunk(z))


def train_panel_net(model, X_team, X_num, y, epochs, lr=1e-3, weight_decay=1e-4,
                     val_team=None, val_num=None, val_y=None, patience=25):
    """Adam + early stopping on an internal validation set the caller
    carves out (never the real holdout -- see each notebook's split cell).
    Returns (train_loss_curve, val_loss_curve); leaves `model` trained
    in-place at its best-validation-loss state.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = torch.nn.MSELoss()
    train_curve, val_curve = [], []
    best_val, best_state, bad_epochs = float("inf"), None, 0
    has_val = val_team is not None and len(val_team) > 0

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(X_team, X_num)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        train_curve.append(float(loss.item()))

        if has_val:
            model.eval()
            with torch.no_grad():
                val_loss = float(loss_fn(model(val_team, val_num), val_y).item())
            val_curve.append(val_loss)
            if val_loss < best_val:
                best_val, best_state, bad_epochs = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    break

    if has_val and best_state is not None:
        model.load_state_dict(best_state)
    return train_curve, val_curve


In [ ]:

train_mask = training_pool["year"] <= TRAIN_END_YEAR
holdout_mask = (featured["year"] >= HOLDOUT_START_YEAR) & (featured["year"] <= HOLDOUT_END_YEAR)
train_df = training_pool[train_mask].copy()
holdout_df = featured[holdout_mask].copy()

rng = np.random.default_rng(42)
val_team_ids = set(rng.choice(team_ids_sorted, size=max(1, len(team_ids_sorted) // 5), replace=False))
fit_df = train_df[~train_df["team_id"].isin(val_team_ids)]
val_df = train_df[train_df["team_id"].isin(val_team_ids)]

feat_mean, feat_std = fit_df[FEATURE_COLS].mean(), fit_df[FEATURE_COLS].std().replace(0, 1.0)
y_mean, y_std = fit_df[TARGETS].mean(), fit_df[TARGETS].std().replace(0, 1.0)


def to_tensors(df):
    team_idx = torch.tensor(df["team_embedding_index"].to_numpy(), dtype=torch.long, device=DEVICE)
    x_num = torch.tensor(((df[FEATURE_COLS] - feat_mean) / feat_std).to_numpy(), dtype=torch.float32, device=DEVICE)
    y = torch.tensor(((df[TARGETS] - y_mean) / y_std).to_numpy(), dtype=torch.float32, device=DEVICE)
    return team_idx, x_num, y


fit_team, fit_num, fit_y = to_tensors(fit_df)
val_team, val_num, val_y = to_tensors(val_df) if len(val_df) else (None, None, None)

model = TeamPanelNet(len(team_ids_sorted), len(FEATURE_COLS), len(TARGETS),
                      embed_dim=EMBED_DIM, hidden=HIDDEN, dropout=DROPOUT).to(DEVICE)
t0 = time.time()
train_curve, val_curve = train_panel_net(model, fit_team, fit_num, fit_y, epochs=EPOCHS, lr=LR,
                                          weight_decay=WEIGHT_DECAY, val_team=val_team, val_num=val_num,
                                          val_y=val_y, patience=EARLY_STOPPING_PATIENCE)
print(f"Trained {len(train_curve)} epochs in {time.time() - t0:.1f}s. Final train loss={train_curve[-1]:.4f}")


## Holdout evaluation + sanity baseline

In [ ]:

model.eval()
with torch.no_grad():
    holdout_team, holdout_num, _ = to_tensors(holdout_df.assign(**{t: 0.0 for t in TARGETS}))
    pred_scaled = model(holdout_team, holdout_num).cpu().numpy()
predictions = pred_scaled * y_std.to_numpy() + y_mean.to_numpy()

holdout_predictions, aggregate_holdout_metrics = [], {}
for i, target in enumerate(TARGETS):
    actual = holdout_df[target].to_numpy()
    pred = predictions[:, i]
    valid = ~np.isnan(actual)
    r2 = float(r2_score(actual[valid], pred[valid])) if valid.sum() >= 2 else None
    rmse = float(np.sqrt(mean_squared_error(actual[valid], pred[valid]))) if valid.sum() >= 2 else None
    aggregate_holdout_metrics[target] = {"r2": r2, "rmse": rmse, "n": int(valid.sum())}
    for (_, row), a, p in zip(holdout_df.iterrows(), actual, pred):
        holdout_predictions.append({"team_id": int(row["team_id"]), "year": int(row["year"]),
                                     "metric": target, "actual": None if np.isnan(a) else float(a),
                                     "predicted": float(p)})
    print(f"{target:>24s}  holdout R2={r2}  RMSE={rmse}  n={valid.sum()}")

from sklearn.linear_model import LinearRegression
baseline_comparison = {}
for target in TARGETS:
    lr_model = LinearRegression().fit(fit_df[FEATURE_COLS], fit_df[target])
    pred = lr_model.predict(holdout_df[FEATURE_COLS])
    actual = holdout_df[target].to_numpy()
    valid = ~np.isnan(actual)
    baseline_comparison[target] = {
        "r2": float(r2_score(actual[valid], pred[valid])) if valid.sum() >= 2 else None,
        "rmse": float(np.sqrt(mean_squared_error(actual[valid], pred[valid]))) if valid.sum() >= 2 else None,
    }
print("Baseline (plain linear regression) holdout R2:", {k: v["r2"] for k, v in baseline_comparison.items()})


## Forward forecast

In [ ]:

def forecast_forward_statcast(model, history_df, team_id, start_year, end_year):
    history = history_df[history_df["team_id"] == team_id].sort_values("year").copy()
    out = []
    for year in range(start_year, end_year + 1):
        last = history.iloc[-1]
        row = {"team_id": team_id, "year": year, "year_norm": (year - STATCAST_ERA_START_YEAR) / 15.0}
        row.update(**{feat: last[feat] for feat in STATCAST_FEATURES})
        row.update(**{f"lag_1_{feat}": last[feat] for feat in STATCAST_FEATURES})
        row.update(regime_flags(year))
        x_num = torch.tensor([[row[c] for c in FEATURE_COLS]], dtype=torch.float32, device=DEVICE)
        x_num = (x_num - torch.tensor(feat_mean[FEATURE_COLS].to_numpy(), dtype=torch.float32, device=DEVICE)) / torch.tensor(feat_std[FEATURE_COLS].to_numpy(), dtype=torch.float32, device=DEVICE)
        team_idx = torch.tensor([TEAM_EMBED_INDEX[team_id]], dtype=torch.long, device=DEVICE)
        with torch.no_grad():
            pred = (model(team_idx, x_num).cpu().numpy()[0] * y_std.to_numpy() + y_mean.to_numpy())
        for i, target in enumerate(TARGETS):
            row[target] = float(pred[i])
        out.append(row)
        history = pd.concat([history, pd.DataFrame([row])], ignore_index=True)
    return out


forward_forecasts = []
for team in ALL_TEAMS:
    for row in forecast_forward_statcast(model, featured, team["team_id"], HOLDOUT_END_YEAR + 1, FORECAST_END_YEAR):
        for target in TARGETS:
            forward_forecasts.append({"team_id": team["team_id"], "year": row["year"], "metric": target,
                                       "predicted": row[target], "ci_lower": None, "ci_upper": None})
print(f"{len(forward_forecasts)} forward forecast rows ({HOLDOUT_END_YEAR + 1}-{FORECAST_END_YEAR}).")


## Results JSON export

Download from `/kaggle/working/results/` after the run, commit into
`notebooks/results/` in the repo, then run
`python scripts/load_team_forecasts.py notebooks/results/<file>.json`.


In [ ]:

import datetime
import subprocess

try:
    git_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
except Exception:
    git_commit = None

run_stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y.%m.%d-%H%M")
results = {
    "schema_version": "1.0",
    "track": TRACK,
    "model_version": f"{TRACK}-{run_stamp}-gpu",
    "run_timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "environment": ENVIRONMENT,
    "git_commit": git_commit,
    "random_seed": 42,
    "hyperparameters": {"hidden_dims": list(HIDDEN), "embedding_dim": EMBED_DIM, "dropout": DROPOUT,
                         "lr": LR, "weight_decay": WEIGHT_DECAY, "epochs_trained": len(train_curve),
                         "early_stopping_patience": EARLY_STOPPING_PATIENCE},
    "targets": TARGETS,
    "feature_list": FEATURE_COLS,
    "teams": [{"team_id": t["team_id"], "abbreviation": t["abbreviation"],
               "embedding_index": TEAM_EMBED_INDEX[t["team_id"]]} for t in ALL_TEAMS],
    "training_window": {"start_year": TRAIN_START_YEAR, "end_year": TRAIN_END_YEAR},
    "holdout_window": {"start_year": HOLDOUT_START_YEAR, "end_year": HOLDOUT_END_YEAR},
    "regime_flags_used": REGIME_FLAG_NAMES,
    "excluded_rows": [{"team_id": r["team_id"], "year": r["year"], "reason": "covid_60_game_season"} for r in excluded_rows],
    "baseline_comparison": baseline_comparison,
    "aggregate_holdout_metrics": aggregate_holdout_metrics,
    "holdout_predictions": holdout_predictions,
    "forward_forecasts": forward_forecasts,
    "loss_curve": {"train": train_curve, "val": val_curve},
    "notes": "",
}

out_dir = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "results"
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/{results['model_version']}.json"
with open(out_path, "w") as fh:
    json.dump(results, fh, indent=2)
print("Wrote", out_path)
